
persons_url='https://data.cityofnewyork.us/api/views/f55k-p6yu/rows.csv?accessType=download'


In [8]:
import pandas as pd
import ssl
import certifi
from urllib.request import urlopen

# Create an SSL context using certifi
ssl_context = ssl.create_default_context(cafile=certifi.where())

# URL of the CSV
crashes_url = 'https://data.cityofnewyork.us/api/views/h9gi-nx95/rows.csv?accessType=download'

# Open the URL with SSL context and pass it to pandas
with urlopen(crashes_url, context=ssl_context) as response:
    df_crashes = pd.read_csv(response, low_memory=False)

# Show the first few rows
df_crashes.head()

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,09/11/2021,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,11/01/2023,1:29,BROOKLYN,11230,40.62179,-73.970024,"(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,NaN,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,06/29/2022,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,09/21/2022,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN


In [9]:
df_crashes.info()


<class 'pandas.DataFrame'>
RangeIndex: 2248025 entries, 0 to 2248024
Data columns (total 29 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   CRASH DATE                     str    
 1   CRASH TIME                     str    
 2   BOROUGH                        str    
 3   ZIP CODE                       str    
 4   LATITUDE                       float64
 5   LONGITUDE                      float64
 6   LOCATION                       str    
 7   ON STREET NAME                 str    
 8   CROSS STREET NAME              str    
 9   OFF STREET NAME                str    
 10  NUMBER OF PERSONS INJURED      float64
 11  NUMBER OF PERSONS KILLED       float64
 12  NUMBER OF PEDESTRIANS INJURED  int64  
 13  NUMBER OF PEDESTRIANS KILLED   int64  
 14  NUMBER OF CYCLIST INJURED      int64  
 15  NUMBER OF CYCLIST KILLED       int64  
 16  NUMBER OF MOTORIST INJURED     int64  
 17  NUMBER OF MOTORIST KILLED      int64  
 18  CONTRIBUTING 

In [10]:
df_crashes.shape


(2248025, 29)

In [11]:
df_crashes.isna().sum().sort_values(ascending=False)


VEHICLE TYPE CODE 5              2238161
CONTRIBUTING FACTOR VEHICLE 5    2237839
VEHICLE TYPE CODE 4              2212187
CONTRIBUTING FACTOR VEHICLE 4    2210833
VEHICLE TYPE CODE 3              2091748
CONTRIBUTING FACTOR VEHICLE 3    2085349
OFF STREET NAME                  1848051
CROSS STREET NAME                 860289
ZIP CODE                          686532
BOROUGH                           686249
ON STREET NAME                    492348
VEHICLE TYPE CODE 2               456005
CONTRIBUTING FACTOR VEHICLE 2     364149
LOCATION                          240635
LONGITUDE                         240635
LATITUDE                          240635
VEHICLE TYPE CODE 1                16733
CONTRIBUTING FACTOR VEHICLE 1       8104
NUMBER OF PERSONS KILLED              31
NUMBER OF PERSONS INJURED             18
COLLISION_ID                           0
CRASH DATE                             0
NUMBER OF MOTORIST KILLED              0
NUMBER OF MOTORIST INJURED             0
NUMBER OF CYCLIS

**Combine CRASH DATE and CRASH TIME into a single datetime column**

In [ ]:
df_crashes['CRASH_DATETIME'] = pd.to_datetime(df_crashes['CRASH DATE'] + ' ' + df_crashes['CRASH TIME'], errors='coerce')

# df_crashes.drop(['CRASH DATE','CRASH TIME'], axis=1, inplace=True)

# Check
df_crashes['CRASH_DATETIME'].head()


0   2021-09-11 02:39:00
1   2022-03-26 11:45:00
2   2023-11-01 01:29:00
3   2022-06-29 06:55:00
4   2022-09-21 13:21:00
Name: CRASH_DATETIME, dtype: datetime64[us]

**clean the crash dataset**

In [13]:
# 1️⃣ Drop mostly-empty or redundant columns
# -----------------------
cols_to_drop = [
    'VEHICLE TYPE CODE 5', 'CONTRIBUTING FACTOR VEHICLE 5',
    'VEHICLE TYPE CODE 4', 'CONTRIBUTING FACTOR VEHICLE 4',
    'VEHICLE TYPE CODE 3', 'CONTRIBUTING FACTOR VEHICLE 3',
    'VEHICLE TYPE CODE 2', 'CONTRIBUTING FACTOR VEHICLE 2',
    'ZIP CODE'  # optional, drop if not needed for filter
]

df_crashes.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [14]:
# Count missing values per column
df_crashes.isna().sum().sort_values(ascending=False)


OFF STREET NAME                  1848051
CROSS STREET NAME                 860289
BOROUGH                           686249
ON STREET NAME                    492348
LATITUDE                          240635
LONGITUDE                         240635
LOCATION                          240635
VEHICLE TYPE CODE 1                16733
CONTRIBUTING FACTOR VEHICLE 1       8104
NUMBER OF PERSONS KILLED              31
NUMBER OF PERSONS INJURED             18
NUMBER OF MOTORIST INJURED             0
COLLISION_ID                           0
NUMBER OF MOTORIST KILLED              0
CRASH DATE                             0
NUMBER OF CYCLIST KILLED               0
NUMBER OF CYCLIST INJURED              0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF PEDESTRIANS INJURED          0
CRASH TIME                             0
CRASH_DATETIME                         0
dtype: int64

In [15]:
# Columns to check
cols_to_check = ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME']

# Count how many times each LOCATION appears
location_counts = df_crashes['LOCATION'].value_counts()

# Locations that appear more than once
locations_multiple = location_counts[location_counts > 1].index

# Filter rows with those locations
rows_multiple_locations = df_crashes[df_crashes['LOCATION'].isin(locations_multiple)]

# Count rows with missing values in any of the columns of interest
num_rows_missing = rows_multiple_locations[cols_to_check].isnull().any(axis=1).sum()

print("Number of rows with shared LOCATION and missing borough/street info:", num_rows_missing)


Number of rows with shared LOCATION and missing borough/street info: 1811258


**the places that have the same location should have the same ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME'] so fill the missing places with the places of the same location**

In [16]:
import pandas as pd

# Columns to fill
cols_to_mode = ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME']

# Filter only the columns we need to reduce memory usage
df_small = df_crashes[['LOCATION'] + cols_to_mode].copy()

# Only consider locations that appear more than once
location_counts = df_small['LOCATION'].value_counts()
locations_multiple = location_counts[location_counts > 1].index
df_multiple = df_small[df_small['LOCATION'].isin(locations_multiple)]

# Compute the mode using a groupby + agg with pd.Series.mode
def safe_mode(x):
    # Exclude NaNs before computing mode
    x_no_nan = x.dropna()
    return x_no_nan.mode()[0] if not x_no_nan.mode().empty else None

mode_per_location = df_multiple.groupby('LOCATION')[cols_to_mode].agg(safe_mode)

# Merge the mode table back to the original dataset on LOCATION
df_crashes = df_crashes.merge(mode_per_location, on='LOCATION', how='left', suffixes=('', '_MODE'))

# Fill missing values only if the mode is not NaN
for col in cols_to_mode:
    mask = df_crashes[col].isnull() & df_crashes[f'{col}_MODE'].notnull()
    df_crashes.loc[mask, col] = df_crashes.loc[mask, f'{col}_MODE']

# Drop the temporary mode columns
df_crashes.drop(columns=[f'{col}_MODE' for col in cols_to_mode], inplace=True)


In [17]:
# Columns to check
cols_to_check = ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME']

# Count how many times each LOCATION appears
location_counts = df_crashes['LOCATION'].value_counts()

# Locations that appear more than once
locations_multiple = location_counts[location_counts > 1].index

# Filter rows with those locations
rows_multiple_locations = df_crashes[df_crashes['LOCATION'].isin(locations_multiple)]

# Count rows with missing values in any of the columns of interest
num_rows_missing = rows_multiple_locations[cols_to_check].isnull().any(axis=1).sum()

print("Number of rows with shared LOCATION and missing borough/street info:", num_rows_missing)


Number of rows with shared LOCATION and missing borough/street info: 1563487


**do the opposite to guess the missing location of the rows
that have the same ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME']**

In [18]:
missing_loc = df_crashes[df_crashes['LATITUDE'].isnull() | df_crashes['LONGITUDE'].isnull()]
print("Rows with missing LAT/LON:", len(missing_loc))


Rows with missing LAT/LON: 240635


In [19]:
cols_group = ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME']


In [20]:
# Only consider rows where LAT/LON exist
valid_loc = df_crashes.dropna(subset=['LATITUDE', 'LONGITUDE'])

# Compute mode LAT/LON per group
mode_loc_per_group = valid_loc.groupby(cols_group)[['LATITUDE', 'LONGITUDE']] \
                     .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)


In [21]:
# Merge mode table back on the street/borough columns
df_crashes = df_crashes.merge(mode_loc_per_group, on=cols_group, how='left', suffixes=('', '_MODE'))

# Fill missing LAT/LON
df_crashes['LATITUDE'] = df_crashes['LATITUDE'].fillna(df_crashes['LATITUDE_MODE'])
df_crashes['LONGITUDE'] = df_crashes['LONGITUDE'].fillna(df_crashes['LONGITUDE_MODE'])

# Drop temporary columns
df_crashes.drop(columns=['LATITUDE_MODE', 'LONGITUDE_MODE'], inplace=True)


In [22]:
missing_after = df_crashes[['LATITUDE', 'LONGITUDE']].isnull().any(axis=1).sum()
print("Number of rows still missing LAT/LON after filling from streets:", missing_after)


Number of rows still missing LAT/LON after filling from streets: 240635


**If multiple crashes happened at nearly the same location, use the most common borough and street names from that location to fill in the missing ones**

In [23]:
cols_street = ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME']
cols_loc = ['LATITUDE', 'LONGITUDE']

# Step 1: Round LAT/LON for clustering
df_crashes['LAT_ROUND'] = df_crashes['LATITUDE'].round(5)  # ~1 m precision
df_crashes['LON_ROUND'] = df_crashes['LONGITUDE'].round(5)
df_crashes['LOC_CLUSTER'] = list(zip(df_crashes['LAT_ROUND'], df_crashes['LON_ROUND']))

# Only clusters that appear more than once
cluster_counts = df_crashes['LOC_CLUSTER'].value_counts()
clusters_multiple = cluster_counts[cluster_counts > 1].index
df_multiple_clusters = df_crashes[df_crashes['LOC_CLUSTER'].isin(clusters_multiple)]

# Compute mode per cluster, excluding NaNs
def safe_mode(x):
    x_no_nan = x.dropna()
    return x_no_nan.mode().iloc[0] if not x_no_nan.mode().empty else None

mode_per_cluster = df_multiple_clusters.groupby('LOC_CLUSTER')[cols_street].agg(safe_mode)

# Merge mode back
df_crashes = df_crashes.merge(mode_per_cluster, on='LOC_CLUSTER', how='left', suffixes=('', '_MODE'))

# Fill missing streets/borough only if the mode is not NaN
for col in cols_street:
    mask = df_crashes[col].isnull() & df_crashes[f'{col}_MODE'].notnull()
    df_crashes.loc[mask, col] = df_crashes.loc[mask, f'{col}_MODE']

# Drop the temporary mode columns
df_crashes.drop(columns=[f'{col}_MODE' for col in cols_street], inplace=True)


**the opposite. If multiple crashes happened on the same street in the same borough, use the most common coordinates for that street to fill in missing location data.**

In [24]:
# Rows where LAT/LON exist
valid_loc = df_crashes.dropna(subset=['LATITUDE', 'LONGITUDE'])

# Compute mode LAT/LON per street combination
mode_loc_per_group = valid_loc.groupby(cols_street)[cols_loc] \
                     .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)

# Merge mode back
df_crashes = df_crashes.merge(mode_loc_per_group, on=cols_street, how='left', suffixes=('', '_MODE'))

# Fill missing LAT/LON
for col in cols_loc:
    df_crashes[col] = df_crashes[col].fillna(df_crashes[f'{col}_MODE'])

df_crashes.drop(columns=[f'{col}_MODE' for col in cols_loc], inplace=True)


In [25]:
df_crashes.isna().sum().sort_values(ascending=False)


OFF STREET NAME                  1327385
CROSS STREET NAME                 574138
ON STREET NAME                    399639
BOROUGH                           275043
LATITUDE                          240635
LONGITUDE                         240635
LOCATION                          240635
LON_ROUND                         240635
LAT_ROUND                         240635
VEHICLE TYPE CODE 1                16733
CONTRIBUTING FACTOR VEHICLE 1       8104
NUMBER OF PERSONS KILLED              31
NUMBER OF PERSONS INJURED             18
CRASH DATE                             0
CRASH_DATETIME                         0
COLLISION_ID                           0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF MOTORIST KILLED              0
NUMBER OF MOTORIST INJURED             0
NUMBER OF CYCLIST KILLED               0
NUMBER OF CYCLIST INJURED              0
CRASH TIME                             0
NUMBER OF PEDESTRIANS INJURED          0
LOC_CLUSTER                            0
dtype: int64

**Step 1: For rows with BOROUGH and ON STREET NAME but missing CROSS STREET NAME, it fills the most common CROSS STREET NAME for that pair.**

**Step 2: For rows with BOROUGH, ON STREET NAME, and CROSS STREET NAME but missing OFF STREET NAME, it fills the most common OFF STREET NAME for that triple.**

**Step 3: For rows with street names but missing BOROUGH, it fills the most common BOROUGH for that**

In [26]:
# Columns to fill
cols_street = ['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME']

# Step 1: Fill CROSS STREET NAME based on BOROUGH + ON STREET NAME
mode_cross = df_crashes.groupby(['BOROUGH', 'ON STREET NAME'])['CROSS STREET NAME'] \
                        .agg(lambda x: x.dropna().mode().iloc[0] if not x.dropna().mode().empty else None)

df_crashes = df_crashes.merge(mode_cross, on=['BOROUGH', 'ON STREET NAME'], how='left', suffixes=('', '_MODE'))
df_crashes['CROSS STREET NAME'] = df_crashes['CROSS STREET NAME'].fillna(df_crashes['CROSS STREET NAME_MODE'])
df_crashes.drop(columns=['CROSS STREET NAME_MODE'], inplace=True)

# Step 2: Fill OFF STREET NAME based on BOROUGH + ON STREET NAME + CROSS STREET NAME
mode_off = df_crashes.groupby(['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME'])['OFF STREET NAME'] \
                     .agg(lambda x: x.dropna().mode().iloc[0] if not x.dropna().mode().empty else None)

df_crashes = df_crashes.merge(mode_off, on=['BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME'], how='left', suffixes=('', '_MODE'))
df_crashes['OFF STREET NAME'] = df_crashes['OFF STREET NAME'].fillna(df_crashes['OFF STREET NAME_MODE'])
df_crashes.drop(columns=['OFF STREET NAME_MODE'], inplace=True)

# Step 3: Fill BOROUGH based on available street info (ON + CROSS + OFF)
mode_borough = df_crashes.groupby(['ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME'])['BOROUGH'] \
                         .agg(lambda x: x.dropna().mode().iloc[0] if not x.dropna().mode().empty else None)

df_crashes = df_crashes.merge(mode_borough, on=['ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME'], how='left', suffixes=('', '_MODE'))
df_crashes['BOROUGH'] = df_crashes['BOROUGH'].fillna(df_crashes['BOROUGH_MODE'])
df_crashes.drop(columns=['BOROUGH_MODE'], inplace=True)

# Optional: check remaining missing
missing_streets = df_crashes[cols_street].isnull().any(axis=1).sum()
print(f"Rows still missing streets/borough: {missing_streets}")


Rows still missing streets/borough: 1494859


In [27]:
df_crashes.isna().sum().sort_values(ascending=False)


OFF STREET NAME                  1125348
CROSS STREET NAME                 549114
ON STREET NAME                    399639
BOROUGH                           274894
LATITUDE                          240635
LONGITUDE                         240635
LOCATION                          240635
LON_ROUND                         240635
LAT_ROUND                         240635
VEHICLE TYPE CODE 1                16733
CONTRIBUTING FACTOR VEHICLE 1       8104
NUMBER OF PERSONS KILLED              31
NUMBER OF PERSONS INJURED             18
CRASH DATE                             0
CRASH_DATETIME                         0
COLLISION_ID                           0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF MOTORIST KILLED              0
NUMBER OF MOTORIST INJURED             0
NUMBER OF CYCLIST KILLED               0
NUMBER OF CYCLIST INJURED              0
CRASH TIME                             0
NUMBER OF PEDESTRIANS INJURED          0
LOC_CLUSTER                            0
dtype: int64

**predict missing ON STREET NAME values using a K-Nearest Neighbors model based on the crash locations’ latitude and longitude**

In [28]:
from sklearn.neighbors import KNeighborsClassifier

# 1️⃣ Training data: rows with known ON STREET NAME and coordinates
train = df_crashes.dropna(subset=['ON STREET NAME', 'LATITUDE', 'LONGITUDE'])

# 2️⃣ Rows to predict: only missing ON STREET NAME AND have valid coordinates
predict_idx = df_crashes['ON STREET NAME'].isnull() & df_crashes['LATITUDE'].notnull() & df_crashes['LONGITUDE'].notnull()
to_predict = df_crashes.loc[predict_idx, ['LATITUDE', 'LONGITUDE']]

# 3️⃣ Train KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train[['LATITUDE', 'LONGITUDE']], train['ON STREET NAME'])

# 4️⃣ Predict ON STREET NAME
df_crashes.loc[predict_idx, 'ON STREET NAME'] = knn.predict(to_predict)


In [29]:
df_crashes.isna().sum().sort_values(ascending=False)


OFF STREET NAME                  1125348
CROSS STREET NAME                 549114
BOROUGH                           274894
LATITUDE                          240635
LONGITUDE                         240635
LOCATION                          240635
LON_ROUND                         240635
LAT_ROUND                         240635
VEHICLE TYPE CODE 1                16733
CONTRIBUTING FACTOR VEHICLE 1       8104
NUMBER OF PERSONS KILLED              31
NUMBER OF PERSONS INJURED             18
CRASH DATE                             0
NUMBER OF MOTORIST KILLED              0
CRASH_DATETIME                         0
COLLISION_ID                           0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF MOTORIST INJURED             0
NUMBER OF CYCLIST KILLED               0
NUMBER OF CYCLIST INJURED              0
CRASH TIME                             0
NUMBER OF PEDESTRIANS INJURED          0
ON STREET NAME                         0
LOC_CLUSTER                            0
dtype: int64

**This code fills missing BOROUGH values by finding another crash on the same ON STREET NAME and copying the BOROUGH from the geographically closest (latitude/longitude) known record.**

In [30]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KDTree

# --- PREP ---

# Keep only rows with latitude/longitude
df_known = df_crashes.dropna(subset=['BOROUGH', 'ON STREET NAME', 'LATITUDE', 'LONGITUDE'])
df_missing = df_crashes[df_crashes['BOROUGH'].isna() &
                        df_crashes['ON STREET NAME'].notna() &
                        df_crashes['LATITUDE'].notna() &
                        df_crashes['LONGITUDE'].notna()]

print("Known with borough:", df_known.shape)
print("Missing borough:", df_missing.shape)

# Output column
df_crashes['BOROUGH_PRED'] = df_crashes['BOROUGH']


Known with borough: (1732496, 24)
Missing borough: (274894, 24)


In [31]:
trees = {}
street_coords = {}
street_borough = {}

for street, group in df_known.groupby('ON STREET NAME'):
    coords = group[['LATITUDE', 'LONGITUDE']].values
    if len(coords) < 1:
        continue
    trees[street] = KDTree(coords)
    street_coords[street] = coords
    street_borough[street] = group['BOROUGH'].values


In [32]:
predictions = []

for idx, row in df_missing.iterrows():
    street = row['ON STREET NAME']

    # skip if street has no known borough data
    if street not in trees:
        continue

    query_point = np.array([[row['LATITUDE'], row['LONGITUDE']]])
    dist, nearest_idx = trees[street].query(query_point, k=1)

    pred_borough = street_borough[street][nearest_idx[0][0]]
    predictions.append((idx, pred_borough))

# Assign predictions back to dataframe
for idx, borough in predictions:
    df_crashes.loc[idx, 'BOROUGH_PRED'] = borough


In [33]:
num_filled = df_crashes['BOROUGH_PRED'].notna().sum() - df_crashes['BOROUGH'].notna().sum()
print(f"Predicted and filled boroughs: {num_filled}")

# Replace BOROUGH with predicted values where NA
df_crashes['BOROUGH'] = df_crashes['BOROUGH_PRED']
df_crashes.drop(columns=['BOROUGH_PRED'], inplace=True)


Predicted and filled boroughs: 269428


In [34]:
df_crashes.isna().sum().sort_values(ascending=False)


OFF STREET NAME                  1125348
CROSS STREET NAME                 549114
LATITUDE                          240635
LONGITUDE                         240635
LOCATION                          240635
LON_ROUND                         240635
LAT_ROUND                         240635
VEHICLE TYPE CODE 1                16733
CONTRIBUTING FACTOR VEHICLE 1       8104
BOROUGH                             5466
NUMBER OF PERSONS KILLED              31
NUMBER OF PERSONS INJURED             18
CRASH DATE                             0
NUMBER OF MOTORIST KILLED              0
CRASH_DATETIME                         0
COLLISION_ID                           0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF MOTORIST INJURED             0
NUMBER OF CYCLIST KILLED               0
NUMBER OF CYCLIST INJURED              0
CRASH TIME                             0
NUMBER OF PEDESTRIANS INJURED          0
ON STREET NAME                         0
LOC_CLUSTER                            0
dtype: int64

**Finds rows with missing OFF STREET NAME then
 Looks for same ON STREET NAME
 and Finds the closest row by latitude/longitude
 Copies its OFF STREET NAME**

In [35]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KDTree

df = df_crashes  # shortcut

df_valid = df.dropna(subset=['LATITUDE', 'LONGITUDE'])


In [36]:
df_valid = df.dropna(subset=['LATITUDE', 'LONGITUDE'])
df_known_off = df_valid.dropna(subset=['OFF STREET NAME'])
df_missing_off = df_valid[df_valid['OFF STREET NAME'].isna() &
                          df_valid['ON STREET NAME'].notna()]


In [37]:
trees = {}
coords_dict = {}
off_values_dict = {}

for street, group in df_known_off.groupby('ON STREET NAME'):
    coords = group[['LATITUDE', 'LONGITUDE']].values
    if len(coords) < 1:
        continue

    trees[street] = KDTree(coords)
    coords_dict[street] = coords
    off_values_dict[street] = group['OFF STREET NAME'].values


In [38]:
pred_off = {}

# Group missing rows by ON STREET NAME
for street, group_missing in df_missing_off.groupby('ON STREET NAME'):

    # Skip if no KDTree for this street
    if street not in trees:
        continue

    # Extract all coordinates for missing rows (batch)
    query_coords = group_missing[['LATITUDE', 'LONGITUDE']].values

    # Batch KDTree query → VERY fast
    distances, indices = trees[street].query(query_coords, k=1)

    # Values to assign
    predicted_values = off_values_dict[street][indices.flatten()]

    # Store predictions mapped by index
    pred_off.update({
        idx: pred for idx, pred in zip(group_missing.index, predicted_values)
    })


In [39]:
df.loc[list(pred_off.keys()), 'OFF STREET NAME'] = list(pred_off.values())


In [40]:
df_crashes.isna().sum().sort_values(ascending=False)


CROSS STREET NAME                549114
LATITUDE                         240635
LONGITUDE                        240635
LOCATION                         240635
LON_ROUND                        240635
LAT_ROUND                        240635
OFF STREET NAME                   25494
VEHICLE TYPE CODE 1               16733
CONTRIBUTING FACTOR VEHICLE 1      8104
BOROUGH                            5466
NUMBER OF PERSONS KILLED             31
NUMBER OF PERSONS INJURED            18
CRASH DATE                            0
NUMBER OF MOTORIST KILLED             0
CRASH_DATETIME                        0
COLLISION_ID                          0
NUMBER OF PEDESTRIANS KILLED          0
NUMBER OF MOTORIST INJURED            0
NUMBER OF CYCLIST KILLED              0
NUMBER OF CYCLIST INJURED             0
CRASH TIME                            0
NUMBER OF PEDESTRIANS INJURED         0
ON STREET NAME                        0
LOC_CLUSTER                           0
dtype: int64

**fill missing CROSS STREET NAME values by finding the nearest crash on the same ON STREET NAME and copying its cross street using geographic distance.**

In [41]:
df_valid = df_crashes.dropna(subset=['LATITUDE', 'LONGITUDE'])

df_known_cross = df_valid.dropna(subset=['CROSS STREET NAME'])
df_missing_cross = df_valid[df_valid['CROSS STREET NAME'].isna() &
                            df_valid['ON STREET NAME'].notna()]


In [42]:
trees_cross = {}
coords_cross = {}
values_cross = {}

for street, group in df_known_cross.groupby('ON STREET NAME'):
    coords = group[['LATITUDE', 'LONGITUDE']].values
    if len(coords) == 0:
        continue
    trees_cross[street] = KDTree(coords)
    coords_cross[street] = coords
    values_cross[street] = group['CROSS STREET NAME'].values


In [43]:
pred_cross = {}

for street, group_missing in df_missing_cross.groupby('ON STREET NAME'):

    # If this street has no known cross street values → skip
    if street not in trees_cross:
        continue

    # All coordinates for missing rows (batch query)
    query_coords = group_missing[['LATITUDE', 'LONGITUDE']].values

    # KDTree batch query
    _, nearest_idx = trees_cross[street].query(query_coords, k=1)

    # predicted values from the closest known rows
    preds = values_cross[street][nearest_idx.flatten()]

    # assign predictions
    pred_cross.update({
        idx: pred for idx, pred in zip(group_missing.index, preds)
    })


In [44]:
df_crashes.loc[list(pred_cross.keys()), 'CROSS STREET NAME'] = list(pred_cross.values())


In [45]:
df_crashes.isna().sum().sort_values(ascending=False)


LATITUDE                         240635
LONGITUDE                        240635
LOCATION                         240635
LON_ROUND                        240635
LAT_ROUND                        240635
OFF STREET NAME                   25494
VEHICLE TYPE CODE 1               16733
CROSS STREET NAME                 12703
CONTRIBUTING FACTOR VEHICLE 1      8104
BOROUGH                            5466
NUMBER OF PERSONS KILLED             31
NUMBER OF PERSONS INJURED            18
CRASH DATE                            0
NUMBER OF MOTORIST KILLED             0
CRASH_DATETIME                        0
COLLISION_ID                          0
NUMBER OF PEDESTRIANS KILLED          0
NUMBER OF MOTORIST INJURED            0
NUMBER OF CYCLIST KILLED              0
NUMBER OF CYCLIST INJURED             0
CRASH TIME                            0
NUMBER OF PEDESTRIANS INJURED         0
ON STREET NAME                        0
LOC_CLUSTER                           0
dtype: int64

**Missing LATITUDE and LONGITUDE are filled in two steps: first, by taking the median coordinates of rows with the same street combination and borough; second, for any remaining missing values, by assigning the coordinates of the nearest known row on the same ON STREET using a KDTree**

In [46]:
fill_combinations = [
    ['ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME', 'BOROUGH'],
    ['ON STREET NAME', 'CROSS STREET NAME', 'BOROUGH'],
    ['ON STREET NAME', 'BOROUGH'],
    ['ON STREET NAME']
]

for cols in fill_combinations:
    # Known rows with coordinates
    df_known = df.dropna(subset=['LATITUDE', 'LONGITUDE'] + cols)
    # Missing rows for these columns
    df_missing = df[df['LATITUDE'].isna() & df['ON STREET NAME'].notna() & df[cols].notna().all(axis=1)]

    if df_missing.empty or df_known.empty:
        continue

    # Median coordinates per combination
    coords_map = df_known.groupby(cols)[['LATITUDE', 'LONGITUDE']].median().reset_index()

    # Merge missing rows with the map
    df_missing_merged = df_missing.merge(coords_map, on=cols, how='left', suffixes=('', '_FILL'))

    # Fill missing LAT/LON
    df.loc[df_missing_merged.index, 'LATITUDE'] = df_missing_merged['LATITUDE'].fillna(df_missing_merged['LATITUDE_FILL'])
    df.loc[df_missing_merged.index, 'LONGITUDE'] = df_missing_merged['LONGITUDE'].fillna(df_missing_merged['LONGITUDE_FILL'])


In [47]:
# Rows still missing coordinates
df_missing_knn = df[df['LATITUDE'].isna() & df['ON STREET NAME'].notna()]

# Rows with known coordinates
df_known_coords = df.dropna(subset=['LATITUDE', 'LONGITUDE', 'ON STREET NAME'])

# Build KDTree per ON STREET NAME
trees = {}
coords_dict = {}
latlon_values = {}

for street, group in df_known_coords.groupby('ON STREET NAME'):
    coords = group[['LATITUDE', 'LONGITUDE']].values
    if len(coords) == 0:
        continue
    trees[street] = KDTree(coords)
    coords_dict[street] = group.index.values
    latlon_values[street] = group[['LATITUDE', 'LONGITUDE']].values

# Vectorized KDTree query per street
pred_coords = {}

for street, group_missing in df_missing_knn.groupby('ON STREET NAME'):
    if street not in trees:
        continue

    query_coords = group_missing[['LATITUDE', 'LONGITUDE']].fillna(0).values  # dummy values
    _, nearest_idx = trees[street].query(query_coords, k=1)

    # Assign predicted LAT/LON
    preds = latlon_values[street][nearest_idx.flatten()]

    pred_coords.update({
        idx: (lat, lon) for idx, (lat, lon) in zip(group_missing.index, preds)
    })

# Apply predictions
df.loc[list(pred_coords.keys()), 'LATITUDE'] = [v[0] for v in pred_coords.values()]
df.loc[list(pred_coords.keys()), 'LONGITUDE'] = [v[1] for v in pred_coords.values()]


In [48]:
df_crashes.isna().sum().sort_values(ascending=False)


LOCATION                         240635
LON_ROUND                        240635
LAT_ROUND                        240635
OFF STREET NAME                   25494
VEHICLE TYPE CODE 1               16733
CROSS STREET NAME                 12703
CONTRIBUTING FACTOR VEHICLE 1      8104
LATITUDE                           7429
LONGITUDE                          7429
BOROUGH                            5466
NUMBER OF PERSONS KILLED             31
NUMBER OF PERSONS INJURED            18
CRASH DATE                            0
NUMBER OF MOTORIST KILLED             0
CRASH_DATETIME                        0
COLLISION_ID                          0
NUMBER OF PEDESTRIANS KILLED          0
NUMBER OF MOTORIST INJURED            0
NUMBER OF CYCLIST KILLED              0
NUMBER OF CYCLIST INJURED             0
CRASH TIME                            0
NUMBER OF PEDESTRIANS INJURED         0
ON STREET NAME                        0
LOC_CLUSTER                           0
dtype: int64

In [49]:
# Fill LOCATION using LATITUDE and LONGITUDE
df['LOCATION'] = df.apply(
    lambda row: f"({row['LATITUDE']}, {row['LONGITUDE']})"
    if pd.notna(row['LATITUDE']) and pd.notna(row['LONGITUDE'])
    else np.nan,
    axis=1
)


In [50]:
df.drop(columns=['LAT_ROUND', 'LON_ROUND'], inplace=True)


In [51]:
cols_to_fill = ['NUMBER OF PERSONS KILLED', 'NUMBER OF PERSONS INJURED']
df[cols_to_fill] = df[cols_to_fill].fillna(0)


In [52]:
df_crashes.isna().sum().sort_values(ascending=False)


OFF STREET NAME                  25494
VEHICLE TYPE CODE 1              16733
CROSS STREET NAME                12703
CONTRIBUTING FACTOR VEHICLE 1     8104
LATITUDE                          7429
LONGITUDE                         7429
LOCATION                          7429
BOROUGH                           5466
CRASH DATE                           0
NUMBER OF CYCLIST KILLED             0
CRASH_DATETIME                       0
COLLISION_ID                         0
NUMBER OF MOTORIST KILLED            0
NUMBER OF MOTORIST INJURED           0
NUMBER OF PEDESTRIANS INJURED        0
NUMBER OF CYCLIST INJURED            0
NUMBER OF PEDESTRIANS KILLED         0
CRASH TIME                           0
NUMBER OF PERSONS KILLED             0
NUMBER OF PERSONS INJURED            0
ON STREET NAME                       0
LOC_CLUSTER                          0
dtype: int64

**



**drop the rows with nan because they will not affect the overall analysis because they are very small compared to the whole dataset**




**





In [53]:
missing_values = ["", " ", "UNKNOWN", "Unspecified", None]
df.replace(missing_values, np.nan, inplace=True)


,CRASH DATE,CRASH TIME,BOROUGH,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,...,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,COLLISION_ID,VEHICLE TYPE CODE 1,CRASH_DATETIME,LOC_CLUSTER
0,09/11/2021,2:39,BROOKLYN,40.693314,-73.933238,"(40.693314, -73.93323799999999)",WHITESTONE EXPRESSWAY,20 AVENUE,PARKING LOT 110-00 ROCKAWAY BOULEVARD,2.0,...,0,0,0,2,0,Aggressive Driving/Road Rage,4455765,Sedan,2021-09-11 02:39:00,"(nan, nan)"
1,03/26/2022,11:45,BROOKLYN,40.592410,-73.788995,"(40.59241, -73.788995)",QUEENSBORO BRIDGE UPPER,HORACE HARDING EXPRESSWAY,PARKING LOT 110-00 ROCKAWAY BOULEVARD,1.0,...,0,0,0,1,0,Pavement Slippery,4513547,Sedan,2022-03-26 11:45:00,"(nan, nan)"
2,11/01/2023,1:29,BROOKLYN,40.687086,-73.933238,"(40.687086, -73.93323799999999)",OCEAN PARKWAY,AVENUE K,AVENUE K,1.0,...,0,0,0,1,0,NaN,4675373,Moped,2023-11-01 01:29:00,"(40.62179, -73.97002)"
3,06/29/2022,6:55,BROOKLYN,40.608475,-73.756480,"(40.608475, -73.75648)",THROGS NECK BRIDGE,HORACE HARDING EXPRESSWAY,PARKING LOT 110-00 ROCKAWAY BOULEVARD,0.0,...,0,0,0,0,0,Following Too Closely,4541903,Sedan,2022-06-29 06:55:00,"(nan, nan)"
4,09/21/2022,13:21,BROOKLYN,40.687086,-73.933238,"(40.687086, -73.93323799999999)",BROOKLYN BRIDGE,HORACE HARDING EXPRESSWAY,PARKING LOT 110-00 ROCKAWAY BOULEVARD,0.0,...,0,0,0,0,0,Passing Too Closely,4566131,Station Wagon/Sport Utility Vehicle,2022-09-21 13:21:00,"(nan, nan)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248020,03/10/2026,1:33,BROOKLYN,40.572933,-74.000310,"(40.572933, -74.00031)",W 35 ST,SURF AVE,2873 WEST 35 STREET,0.0,...,0,0,0,0,0,Lost Consciousness,4884425,Station Wagon/Sport Utility Vehicle,2026-03-10 01:33:00,"(40.57293, -74.00031)"
2248021,03/10/2026,9:40,BRONX,40.851833,-73.841830,"(40.851833, -73.84183)",BASSETT AVENUE,WILKINSON AVENUE,1500 BASSETT AVE,1.0,...,0,0,0,0,0,NaN,4884484,Standing S,2026-03-10 09:40:00,"(40.85183, -73.84183)"
2248022,03/10/2026,22:10,QUEENS,40.688293,-73.831856,"(40.688293, -73.831856)",112 ST,101 AVE,PARKING LOT 112-11 LIBERTY AVENUE,1.0,...,0,0,0,0,0,NaN,4884517,Sedan,2026-03-10 22:10:00,"(40.68829, -73.83186)"
2248023,03/10/2026,15:02,BROOKLYN,40.706696,-73.923280,"(40.706696, -73.92328)",WYCKOFF AVENUE,WYCKOFF AVENUE,8 WYCKOFF AVE,0.0,...,0,0,0,0,0,Other Vehicular,4884398,Station Wagon/Sport Utility Vehicle,2026-03-10 15:02:00,"(40.7067, -73.92328)"


In [54]:
cols_important = [
    'OFF STREET NAME', 'VEHICLE TYPE CODE 1', 'CROSS STREET NAME',
    'CONTRIBUTING FACTOR VEHICLE 1', 'BOROUGH'
]

df = df.dropna(subset=cols_important)


In [55]:
# Define rows with missing street/borough info
cols_check = ['OFF STREET NAME', 'VEHICLE TYPE CODE 1', 'CROSS STREET NAME',
              'CONTRIBUTING FACTOR VEHICLE 1', 'BOROUGH']

# Strip spaces and treat empty strings as missing
df_clean = df[~df[cols_check].apply(lambda x: x.str.strip() == '').any(axis=1)].copy()


In [56]:
df_crashes.isna().sum().sort_values(ascending=False)


CONTRIBUTING FACTOR VEHICLE 1    759042
VEHICLE TYPE CODE 1               36674
OFF STREET NAME                   25494
CROSS STREET NAME                 12703
LATITUDE                           7429
LONGITUDE                          7429
LOCATION                           7429
BOROUGH                            5466
CRASH DATE                            0
NUMBER OF CYCLIST KILLED              0
CRASH_DATETIME                        0
COLLISION_ID                          0
NUMBER OF MOTORIST KILLED             0
NUMBER OF MOTORIST INJURED            0
NUMBER OF PEDESTRIANS INJURED         0
NUMBER OF CYCLIST INJURED             0
NUMBER OF PEDESTRIANS KILLED          0
CRASH TIME                            0
NUMBER OF PERSONS KILLED              0
NUMBER OF PERSONS INJURED             0
ON STREET NAME                        0
LOC_CLUSTER                           0
dtype: int64

In [57]:
df_crashes.shape


(2248025, 22)

**handling oultliers for the location, number of injuries ,and people died**

In [58]:
import pandas as pd

# Make a copy to be safe
df_clean = df.copy()

# Define logical maximums
max_values = {
    'NUMBER OF PERSONS KILLED': 5,
    'NUMBER OF PERSONS INJURED': 5,
    'NUMBER OF PEDESTRIANS KILLED': 2,
    'NUMBER OF PEDESTRIANS INJURED': 2,
    'NUMBER OF CYCLIST KILLED': 2,
    'NUMBER OF CYCLIST INJURED': 2,
    'NUMBER OF MOTORIST KILLED': 2,
    'NUMBER OF MOTORIST INJURED': 2
}

# Clean injury/death columns
for col, max_val in max_values.items():

    # Replace negative values with 0
    df_clean[col] = df_clean[col].clip(lower=0)

    # Cap extremely large values to logical maximum
    df_clean[col] = df_clean[col].clip(upper=max_val)

    # Ensure integer type
    df_clean[col] = df_clean[col].astype(int)


# Latitude/Longitude bounds
lat_min, lat_max = 40.49, 40.92
lon_min, lon_max = -74.27, -73.68

mean_lat = df_clean.loc[df_clean['LATITUDE'].between(lat_min, lat_max), 'LATITUDE'].mean()
mean_lon = df_clean.loc[df_clean['LONGITUDE'].between(lon_min, lon_max), 'LONGITUDE'].mean()

df_clean['LATITUDE'] = df_clean['LATITUDE'].apply(
    lambda x: mean_lat if not (lat_min <= x <= lat_max) else x
)

df_clean['LONGITUDE'] = df_clean['LONGITUDE'].apply(
    lambda x: mean_lon if not (lon_min <= x <= lon_max) else x
)

print("Negative values fixed, large values capped, integers preserved.")

Negative values fixed, large values capped, integers preserved.


In [59]:
numeric_cols = [
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED'
]

# Final safety cleaning
for col in numeric_cols:

    # Replace negative values with 0
    df_clean[col] = df_clean[col].clip(lower=0)

    # Ensure integer values
    df_clean[col] = df_clean[col].astype(int)

In [68]:
# Sum killed and injured by category
pedestrian_injured = df_crashes['NUMBER OF PEDESTRIANS INJURED'].sum()
pedestrian_killed = df_crashes['NUMBER OF PEDESTRIANS KILLED'].sum()

cyclist_injured = df_crashes['NUMBER OF CYCLIST INJURED'].sum()
cyclist_killed = df_crashes['NUMBER OF CYCLIST KILLED'].sum()

motorist_injured = df_crashes['NUMBER OF MOTORIST INJURED'].sum()
motorist_killed = df_crashes['NUMBER OF MOTORIST KILLED'].sum()

# Display results
print(f"Pedestrians → Injured: {pedestrian_injured}, Killed: {pedestrian_killed}")
print(f"Cyclists    → Injured: {cyclist_injured}, Killed: {cyclist_killed}")
print(f"Motorists   → Injured: {motorist_injured}, Killed: {motorist_killed}")

Pedestrians → Injured: 135708, Killed: 1779
Cyclists    → Injured: 65879, Killed: 286
Motorists   → Injured: 531344, Killed: 1446


In [60]:
df_crashes.shape


(2248025, 22)

In [61]:
# Convert CRASH DATE to datetime
df_clean['CRASH DATE'] = pd.to_datetime(df_clean['CRASH DATE'], errors='coerce')

# Convert CRASH_DATETIME to datetime if not already
df_clean['CRASH_DATETIME'] = pd.to_datetime(df_clean['CRASH_DATETIME'], errors='coerce')


In [62]:
string_cols = [
    'BOROUGH', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME',
    'CONTRIBUTING FACTOR VEHICLE 1', 'VEHICLE TYPE CODE 1', 'LOC_CLUSTER'
]

for col in string_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.upper()
    df_clean[col] = df_clean[col].replace({'': pd.NA, 'NAN': pd.NA})


In [63]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows.")


Removed 0 duplicate rows.


In [64]:
df_crashes.shape


(2248025, 22)

In [65]:
df.to_csv('cleaned_crashes.csv', index=False)
